# 08 · Uncertainty, resolution, and synthetic tests

A colorful slip map is easy to over-read. It is an *estimate* built from noisy,
incomplete data, not a photograph of the fault. This chapter gives you the
tools to ask harder questions of a result: how much would the estimate wobble
if the data were re-measured, how much does the inversion blur the true slip,
and where can the data actually see anything at all. These are the questions
that separate a defensible slip model from a pretty picture.

**Learning objectives**

By the end of the chapter, you will be able to:

- read per-patch slip uncertainty and know what it does and does not include
- interpret model resolution, both its diagonal and a full averaging kernel
- design and read a checkerboard recovery test
- propagate slip uncertainty into a seismic-moment uncertainty

**Prerequisites:** Chapters 03 and 04
**Estimated time:** 60–90 minutes

GeoDef's axes, signs, and units are summarized in
[the conventions guide](../docs/conventions.md). New terms are introduced in
words here and collected in [the glossary](../docs/glossary.md).

## 1. Two different questions

"How good is this slip model?" hides two separate questions that must not be
confused.

- **Uncertainty** asks: if we collected the data again, with fresh noise, how
  much would the estimate move? It is about *repeatability*.
- **Resolution** asks: does the inversion return the true slip, or a blurred,
  spatially averaged version of it? It is about *fidelity*.

A patch can score well on one and poorly on the other. Strong smoothing, for
instance, can make a patch's estimate very repeatable (low uncertainty) while
guaranteeing it is a broad average of its neighbors (poor resolution). We need
both numbers to describe a result honestly.

## 2. Rebuild the shared scenario and solve

We reuse the megathrust, true slip, GNSS network, and 1 cm noise from
Chapter 03, and run one smoothed inversion to study.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

%matplotlib inline

rng = np.random.default_rng(0)

In [ ]:
# The shared megathrust, true slip, GNSS network, and 1 cm noise.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,
    strike=315.0, dip=15.0,
    length=180e3, width=90e3,
    n_length=12, n_width=6,
)
N = fault.n_patches
n_length, n_width = fault.grid_shape
along = np.arange(N) % n_length
down = np.arange(N) // n_length
bump = np.exp(-(((along - (n_length - 1) / 2) / 3.0) ** 2
                + ((down - n_width / 2) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

grid_lon, grid_lat = np.meshgrid(np.linspace(98.5, 101.5, 8),
                                 np.linspace(-3.6, -0.4, 8))
station_lon, station_lat = grid_lon.ravel(), grid_lat.ravel()
n_stations = station_lon.size

G_full = fault.greens_matrix(station_lat, station_lon)
sigma = 0.01
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, 3 * n_stations)
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=d_obs[0::3], north=d_obs[1::3], up=d_obs[2::3],
    sigma_east=np.full(n_stations, sigma),
    sigma_north=np.full(n_stations, sigma),
    sigma_up=np.full(n_stations, sigma),
)
result = geodef.solve(fault, gnss, components="dip",
                      regularization="laplacian", regularization_strength=1.0)

## 3. Per-patch uncertainty

`geodef.invert.model_uncertainty` returns the 1-sigma slip error for each
patch: how much that patch's estimate would typically move under fresh noise.
Deep, poorly sampled patches, far from the stations, carry the largest
uncertainty.

In [ ]:
uncertainty = geodef.invert.model_uncertainty(result, fault, gnss)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
geodef.plot.slip(fault, result.dip_slip, ax=ax1, title="Recovered slip",
                 colorbar_label="Slip (m)")
geodef.plot.uncertainty(fault, uncertainty, ax=ax2, title="1-sigma uncertainty")
plt.tight_layout()

One caution: this uncertainty is *conditional on the model setup*. It reflects
the data noise and the chosen regularization, but not the possibility that the
fault geometry, the elastic properties, or the smoothing choice are themselves
wrong. Those larger uncertainties are the subject of Chapter 13. Treat this map
as a lower bound, not the whole story.

## 4. Resolution: the diagonal

The **model resolution matrix** $R$ answers the fidelity question. It relates
the recovered slip to the truth by $\hat{\mathbf{m}} = R\,\mathbf{m}_{\text{true}}$
(on average). If $R$ were the identity matrix, each patch would report its own
true slip and nothing else: perfect resolution. In practice $R$ smears the
truth, and its structure tells us how.

The quickest summary is the diagonal of $R$. A value near 1 marks a
well-resolved patch (typically shallow, beneath the network); a value near 0
marks a patch the data barely constrain.

In [ ]:
R = geodef.invert.model_resolution(result, fault, gnss)   # full (N, N) matrix
resolution_diagonal = np.diag(R)
geodef.plot.resolution(fault, resolution_diagonal, vmin=0, vmax=1,
                       title="Model resolution (diagonal of R)")
print(f"resolution ranges from {resolution_diagonal.min():.2f} "
      f"to {resolution_diagonal.max():.2f}")

## 5. Resolution: reading one full row

The diagonal alone hides something important: *where* a patch's leaked
amplitude comes from. That information lives in the off-diagonal entries. One
**row** of $R$ is a single patch's **averaging kernel**: it shows how every
true patch contributes to that one recovered patch.

Let us pull out the row for a well-resolved patch near the center and un-flatten
it into a map. A sharp spike on the target patch would mean clean resolution; a
broad blob means the recovered value is really an average over a neighborhood.

In [ ]:
target = fault.patch_index(strike_idx=6, dip_idx=1)   # a shallow central patch
kernel_row = R[target]                                # its averaging kernel
print(f"kernel peaks at patch {kernel_row.argmax()} (the target is {target})")

In [ ]:
# Mark the target patch so we can see where the kernel is centered.
target_marker = np.zeros(N)
target_marker[target] = 1.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
geodef.plot.slip(fault, target_marker, ax=ax1, cmap="Greys",
                 colorbar=False, title=f"Target patch {target}")
geodef.plot.slip(fault, kernel_row, ax=ax2, cmap="viridis",
                 colorbar_label="kernel weight",
                 title="Averaging kernel (row of R)")
plt.tight_layout()

The kernel is centered on the target but spread across its neighbors: the
recovered slip there is a local average, not a point sample. This is why two
patches can both show high diagonal resolution yet still blur into each other.
The diagonal tells you *how much* of a patch is resolved; the row tells you
*from where* the rest leaked in.

## 6. A checkerboard test

The most intuitive recoverability check is the **checkerboard test**: paint a
known alternating pattern of high and low slip on the fault, generate synthetic
data from it, invert, and see where the checkers survive. Sharp checkers mark
well-resolved regions; smeared or vanished checkers mark poorly resolved ones.

Because we control the truth, this is an honest test of *this specific
wavelength*. Build the checkerboard, forward-model it with fresh noise, and
invert.

In [ ]:
checker = 1.0 + 0.9 * ((-1.0) ** along) * ((-1.0) ** down)   # alternating high/low
m_check = np.concatenate([np.zeros(N), checker])
d_check = G_full @ m_check + rng.normal(0.0, sigma, 3 * n_stations)
gnss_check = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=d_check[0::3], north=d_check[1::3], up=d_check[2::3],
    sigma_east=np.full(n_stations, sigma),
    sigma_north=np.full(n_stations, sigma),
    sigma_up=np.full(n_stations, sigma),
)
recovered = geodef.solve(fault, gnss_check, components="dip",
                         regularization="laplacian", regularization_strength=0.3)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
geodef.plot.slip(fault, checker, ax=ax1, title="Input checkerboard",
                 colorbar_label="Slip (m)")
geodef.plot.slip(fault, recovered.dip_slip, ax=ax2, title="Recovered",
                 colorbar_label="Slip (m)")
plt.tight_layout()

The checkers survive best where the resolution map of Section 4 was high
(shallow, under the network) and wash out where it was low (deep). The two
diagnostics agree, as they should; the checkerboard is a visual, wavelength-
specific view of the same resolution the matrix $R$ describes.

## 7. Uncertainty in the seismic moment

Finally, a scalar summary with its own error bar. The seismic moment
$M_0 = \mu \sum_k s_k A_k$ is a simple weighted sum of the patch slips, so its
uncertainty follows directly from the slip uncertainty. The cleanest way to
propagate it is to draw many slip models from the posterior distribution and
look at the spread of magnitudes they imply.

`model_covariance` gives the full slip covariance; `rng.multivariate_normal`
draws samples from it. We clip each sample to non-negative slip before
computing its magnitude, since negative slip has no moment.

In [ ]:
Cm = geodef.invert.model_covariance(result, fault, gnss)
samples = rng.multivariate_normal(result.dip_slip, Cm, size=500)
magnitudes = np.array([fault.magnitude(np.clip(s, 0, None)) for s in samples])
print(f"Mw = {magnitudes.mean():.2f} +/- {magnitudes.std():.2f}")

The result is a magnitude with a stated uncertainty, rather than a single
number pretending to be exact. As with the per-patch uncertainty, this spread
reflects data noise and regularization only, not errors in the fault geometry
or elastic model.

**Checkpoint.** A patch shows a high diagonal resolution value but its
averaging kernel (its row of $R$) is broad. Is the patch well resolved?

<details><summary>Answer</summary>

Not really. A high diagonal value says the patch contributes strongly to its
own recovered estimate, but a broad kernel says that estimate also mixes in a
lot of its neighbors. The diagonal can look reassuring while the kernel reveals
substantial blurring, which is exactly why you should inspect rows, not just
the diagonal.

</details>

## Checkpoint questions

**Can a patch have low uncertainty and low resolution at the same time?**

<details><summary>Answer</summary>

Yes. Strong regularization can pin a patch to a repeatable value (low
uncertainty) that is really a broad average of its surroundings (low
resolution). Precision and fidelity are different properties.

</details>

**What does a single row of the resolution matrix show?**

<details><summary>Answer</summary>

How every true patch contributes to one recovered patch: that patch's
averaging kernel. A narrow kernel means clean local resolution; a broad kernel
means the recovered value is a spatial average.

</details>

**What does a checkerboard test fail to test?**

<details><summary>Answer</summary>

Everything outside the one wavelength and pattern you chose: other slip scales,
and any error in the fault geometry, the elastic model, or the noise
covariance. A clean checkerboard recovery is necessary evidence of resolution,
not a full validation of the model.

</details>

## Common mistakes

- **Reporting only the diagonal of $R$.** It hides where leaked amplitude comes
  from. Inspect a few full rows before claiming a region is resolved.
- **Calling the conditional uncertainty the total uncertainty.** The maps here
  omit geometry and elastic-model error, so they are lower bounds. Chapter 13
  addresses the rest.
- **Choosing the checker size after seeing the recovery.** Picking the
  wavelength that happens to recover well turns an honest test into a
  presentation. Fix the test design before looking at the result.

## Recap

- Uncertainty measures repeatability under noise; resolution measures how much
  the inversion averages the truth. They are distinct and both are needed.
- The diagonal of $R$ summarizes resolution; a full row is a patch's averaging
  kernel and shows spatial leakage.
- A checkerboard test is a visual, wavelength-specific view of resolution.
- Moment uncertainty propagates from the slip covariance by sampling the
  posterior.

This chapter closes the beginner sequence on linear slip inversion. The later
chapters relax the fixed-geometry assumption (09, 10), change the fault shape
(11), model the interseismic case (12), confront model error head-on (13), and
build a full Bayesian treatment (14).

## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are in `solutions/08_uncertainty_and_resolution_solutions.ipynb`.

1. **Two wavelengths, two strengths.** Run the checkerboard test at a coarse
   and a fine checker size, each at two smoothing strengths, and tabulate where
   the checkers survive.
2. **A migrating spike.** Place a single spike of slip and move it down dip
   until the inversion can no longer localize it. How deep does resolution give
   out?
3. **Moment error.** Propagate the slip covariance into $M_0$ and $M_w$ as in
   Section 7, and compare the spread at two smoothing strengths.
4. **Challenge: approach $R = I$.** In a small, well-sampled problem, reduce the
   noise and the regularization together and watch the resolution matrix
   approach the identity.

## Further reading

- Backus, G., and Gilbert, F. (1968), "The resolving power of gross Earth
  data," *Geophysical Journal International*, 16(2), 169–205, the origin of
  resolution and averaging kernels.
- Menke, W. (2018), *Geophysical Data Analysis: Discrete Inverse Theory*,
  4th ed., on model covariance and resolution.
- Funning, G. J., et al. (2005), on geodetic resolution testing in practice.
- [GeoDef inversion documentation](../docs/invert.md) for the
  `model_uncertainty`, `model_resolution`, and `model_covariance` interfaces.
- The next course chapter is `09_nonlinear_geometry.ipynb`.